In [1]:
import numpy as np
import pandas as pd
from pandas import Series, DataFrame

In [2]:
df = pd.read_csv(
    '../../../data/nyc_taxi_2019-01.csv',
     engine = 'pyarrow',
    dtype_backend = 'pyarrow',
    usecols = ['passenger_count', 'trip_distance', 'total_amount'] 
)

In [3]:
len(df)

7667792

In [4]:
# For each number of passengers, find the mean cost of a taxi ride. Sort this
# result from lowest (i.e., cheapest) to highest (i.e., most expensive).

df.groupby('passenger_count').mean().sort_values(by = 'total_amount')

,trip_distance,total_amount
passenger_count,,
6,2.842335,15.437892
5,2.865741,15.54694
3,2.840698,15.604015
1,2.779088,15.609601
4,2.853084,15.650307
2,2.880572,15.831294
0,2.651561,18.663658
9,1.486667,31.094444
7,2.561579,48.278421


In [5]:
df.groupby('passenger_count').agg('mean').sort_values(by = 'total_amount')

,trip_distance,total_amount
passenger_count,,
6,2.842335,15.437892
5,2.865741,15.54694
3,2.840698,15.604015
1,2.779088,15.609601
4,2.853084,15.650307
2,2.880572,15.831294
0,2.651561,18.663658
9,1.486667,31.094444
7,2.561579,48.278421


In [6]:
# Sort the results again by increasing the number of passengers.
df.groupby('passenger_count').mean().sort_index()

,trip_distance,total_amount
passenger_count,,
0,2.651561,18.663658
1,2.779088,15.609601
2,2.880572,15.831294
3,2.840698,15.604015
4,2.853084,15.650307
5,2.865741,15.54694
6,2.842335,15.437892
7,2.561579,48.278421
8,3.142759,64.105517


In [7]:
# Create a new column, trip_distance_group, in which the values are
# short (< 2 miles),
# medium (>=2 miles and <= 10 miles), and
# long (> 10 miles).

df['trip_distance_group'] = 'medium'
df.loc[df['trip_distance'] < 2, 'trip_distance_group'] = 'short'
df.loc[df['trip_distance'] > 10, 'trip_distance_group'] = 'long'
df.sample(10)

,passenger_count,trip_distance,total_amount,trip_distance_group
2340452,1,0.77,8.16,short
4747673,1,9.16,40.87,medium
440410,1,0.8,7.8,short
748136,1,2.5,14.8,medium
4132188,1,3.21,17.88,medium
2358116,1,0.81,8.19,short
2406951,3,0.65,4.8,short
772941,1,0.52,8.3,short
4691373,1,3.1,14.3,medium
3120030,1,12.59,58.8,long


In [8]:
# What is the average number of passengers per trip length category?
# Sort this result from highest (most passengers) to lowest (fewest passengers).

df.groupby('trip_distance_group')['passenger_count'].mean()

# NOTE: for my future self, I completely forgot this, but it could have been done using pd.cut as well.
# df['trip_distance_group'] = pd.cut(df['trip_distance'], 
#                                    [df['trip_distance'].min(), 2, 10, 
#                                     df['trip_distance'].max()],
#                                   labels=['short', 'medium', 'long'])
# df.groupby('trip_distance_group')['passenger_count'].mean().sort_values(ascending=False)

trip_distance_group
long      1.590035
medium    1.576764
short     1.559943
Name: passenger_count, dtype: double[pyarrow]

In [9]:
df.groupby('trip_distance_group').mean().sort_values(by = 'passenger_count', ascending = False)

,passenger_count,trip_distance,total_amount
trip_distance_group,,,
long,1.590035,15.549428,57.800367
medium,1.576764,3.927226,20.009066
short,1.559943,1.072269,9.652103


#### Beyond the exercise_1
##### Create a single data frame containing rides from both January 2019 and January 2020, with a column year indicating which year the ride comes from. Use groupby to compare the average cost of a taxi in January from each of these two years.

In [10]:

df_2019 = pd.read_csv(
    '../../../data/nyc_taxi_2019-01.csv',
     engine = 'pyarrow',
    dtype_backend = 'pyarrow',
    usecols = ['passenger_count', 'trip_distance', 'total_amount'] 
)
df_2020 = pd.read_csv(
    '../../../data/nyc_taxi_2020-01.csv',
     engine = 'pyarrow',
    dtype_backend = 'pyarrow',
    usecols = ['passenger_count', 'trip_distance', 'total_amount'] 
)

In [11]:
df_2019['year'] = 2019
df_2020['year'] = 2020
df_january = pd.concat([df_2019, df_2020])


In [12]:
df_january.groupby('year').mean()['total_amount']

year
2019    15.682222
2020    18.663149
Name: total_amount, dtype: double[pyarrow]

#### Beyond the exercise_2
##### Create a two-level grouping, first by `year` and then by `passenger_count`.

In [13]:
df_january.groupby(['year', 'passenger_count'])[['total_amount']].mean()

total_amount
year passenger_count              
2019 0                   18.663658
     1                   15.609601
     2                   15.831294
     3                   15.604015
     4                   15.650307
     5                    15.54694
     6                   15.437892
     7                   48.278421
     8                   64.105517
     9                   31.094444
2020 0                   18.059724
     1                    18.34311
     2                   19.050504
     3                   18.736862
     4                   19.128092
     5                   18.234443
     6                   18.367962
     7                   71.143103
     8                   58.197059
     9                   81.244211

#### Beyond the exercise_3
##### The `corr` method allows us to see how strongly two columns correlate with one another. Use `corr` and then `sort_values` to find which columns have the highest correlation.

In [14]:
df_january.corr()

,passenger_count,trip_distance,total_amount,year
passenger_count,1.000000,0.008974,-0.000136,-0.021602
trip_distance,0.008974,1.000000,0.004331,0.001140
total_amount,-0.000136,0.004331,1.000000,0.007657
year,-0.021602,0.001140,0.007657,1.000000


In [15]:
df_january.corr().sort_values(by = 'passenger_count')

,passenger_count,trip_distance,total_amount,year
year,-0.021602,0.001140,0.007657,1.000000
total_amount,-0.000136,0.004331,1.000000,0.007657
trip_distance,0.008974,1.000000,0.004331,0.001140
passenger_count,1.000000,0.008974,-0.000136,-0.021602
